# Gesture Arm — Benchmark Analysis

Reproduces the evaluation results from the paper:
> *A Low-Cost Multimodal Gesture Control System with LSTM-Based Temporal Stabilization for Real-Time Robotics*

**Paper Section VI metrics:**
- **S** = Control stability variance: `(1/T) Σ (u_t − ū)²`  — lower is better
- **L** = End-to-end latency: `t_actuation − t_capture` (ms)  — lower is better

Run `python scripts/collect.py` then `python scripts/train.py` before executing this notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.2)
plt.rcParams['figure.dpi'] = 120

LOG = Path('../data/metrics_log.csv')
assert LOG.exists(), f'Run the control system first to generate {LOG}'

df = pd.read_csv(LOG)
print(f'Loaded {len(df):,} frames  |  methods: {df.method.unique()}')
df.head()

## 1. Stability variance S by method

In [ ]:
def rolling_stability(group, window=100):
    """S = rolling variance of servo command vector."""
    servo_cols = ['servo_x', 'servo_y', 'servo_z']
    return (
        group[servo_cols]
        .rolling(window, min_periods=10)
        .apply(lambda x: np.var(x), raw=True)
        .mean(axis=1)
    )

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Rolling S over time
ax = axes[0]
for method, grp in df.groupby('method'):
    s = rolling_stability(grp)
    ax.plot(s.values, label=method, linewidth=1.5, alpha=0.85)
ax.set_title('Rolling stability variance S (lower = smoother)')
ax.set_xlabel('Frame')
ax.set_ylabel('S')
ax.legend()

# Boxplot comparison
ax = axes[1]
stability_per_method = {
    m: rolling_stability(g).dropna().values
    for m, g in df.groupby('method')
}
ax.boxplot(stability_per_method.values(), labels=stability_per_method.keys(), patch_artist=True)
ax.set_title('Stability distribution by method')
ax.set_ylabel('S')

plt.tight_layout()
plt.savefig('../docs/stability_comparison.png', bbox_inches='tight')
plt.show()

## 2. Latency L distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for method, grp in df.groupby('method'):
    ax.hist(grp['latency_ms'], bins=60, alpha=0.65, label=method, density=True)
ax.axvline(df['latency_ms'].median(), color='k', linestyle='--', linewidth=1, label='Overall median')
ax.set_title('End-to-end latency L distribution')
ax.set_xlabel('Latency (ms)')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.savefig('../docs/latency_distribution.png', bbox_inches='tight')
plt.show()

## 3. Summary table (paper Table I)

In [ ]:
servo_cols = ['servo_x', 'servo_y', 'servo_z']

rows = []
for method, grp in df.groupby('method'):
    mean_arr = grp[servo_cols].values
    mean_vec = mean_arr.mean(axis=0)
    S        = float(np.mean(np.sum((mean_arr - mean_vec)**2, axis=1)))
    rows.append({
        'Method':           method,
        'N frames':         len(grp),
        'S (stability ↓)':  round(S, 4),
        'L mean (ms) ↓':   round(grp['latency_ms'].mean(), 2),
        'L p95 (ms)':       round(grp['latency_ms'].quantile(0.95), 2),
        'L median (ms)':    round(grp['latency_ms'].median(), 2),
    })

summary = pd.DataFrame(rows).set_index('Method')
print(summary.to_markdown())
summary

## 4. Servo trajectory — LSTM vs baseline

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=False)
axis_labels = ['Servo X (horizontal)', 'Servo Y (vertical)', 'Servo Z (grip)']
colors = {'lstm': '#2563eb', 'baseline': '#d97706', 'baseline (warming)': '#9ca3af'}

for ax, col, label in zip(axes, servo_cols, axis_labels):
    for method, grp in df.groupby('method'):
        ax.plot(
            grp[col].values[:500],
            label=method,
            linewidth=1.2,
            alpha=0.8,
            color=colors.get(method, '#6b7280'),
        )
    ax.set_ylabel(label)
    ax.legend(loc='upper right', fontsize=9)

axes[-1].set_xlabel('Frame (first 500)')
fig.suptitle('Servo trajectory: LSTM stabilized vs baseline (first 500 frames)', fontsize=13)
plt.tight_layout()
plt.savefig('../docs/trajectory_comparison.png', bbox_inches='tight')
plt.show()